# Equation Verification Notebook

Verifies every analytical formula in the simulator against simulation outputs.
Each section ends with a **PASS / FAIL** summary.

## Equations under test

| # | Name | Formula |
|---|------|---------|
| 1 | Success probability | `p = q(1-q)^(n-1)` |
| 2 | Service rate (no sleep) | `μ = p` |
| 3 | Service rate (with sleep) | `μ = p / (1 + tw·λ/(1-λ))` — paper Eq. 12 |
| 4 | Mean delay (M/M/1 approx) | `T̄ = 1/(μ − λ)` |
| 5 | Mean queue length (Little's Law) | `L̄ = λ/(μ − λ)` |
| 6 | Lifetime projection | linear extrapolation from energy rate |

## Key modelling notes

### Note 1 — Two definitions of "success probability"

| Field | Value | Unit |
|---|---|---|
| `empirical_success_prob` | `total_successes / total_slots` | **network throughput S** |
| `empirical_service_rate` | `total_successes / total_active_node_slots` | **per-node rate ≈ p** |

Compare `empirical_service_rate` against analytical `p = q(1-q)^(n-1)`.  
Do **not** compare `empirical_success_prob` (throughput) against `p` — they differ by `n × active_fraction`.

### Note 2 — Effective n in unsaturated regime

The paper formula `p = q(1-q)^(n-1)` assumes all `n` nodes are always contending (saturated).  
In unsaturated operation, only fraction `α` of nodes are ACTIVE, so  
`p_eff = q(1-q)^(n_eff-1)` where `n_eff = n × α`  
gives **much** better agreement with simulated service rates (error ≈5% vs ≈100% for full `n`).

### Note 3 — Lifetime and ts (idle timer)

GENERIC_LOW profile: PI = 2 mW, PS = 0.005 mW (idle is 400× more expensive than sleep).  
Longer ts → more time in IDLE → **shorter** battery lifetime (not longer).

**Date:** March 2026

In [ ]:
# Setup
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.simulator import Simulator, SimulationConfig
from src.metrics import MetricsCalculator
from src.node import Node
from src.power_model import PowerModel, PowerProfile

plt.style.use('seaborn-v0_8-darkgrid')

PASS = '\033[92m[PASS]\033[0m'
FAIL = '\033[91m[FAIL]\033[0m'

def check(label, passed, detail=''):
    tag = PASS if passed else FAIL
    msg = f'{tag} {label}'
    if detail:
        msg += f'  →  {detail}'
    print(msg)
    return passed

def run_sim(n=20, q=0.05, lam=0.01, ts=10, tw=5, energy=8000, slots=50_000, seed=42):
    cfg = SimulationConfig(
        n_nodes=n, arrival_rate=lam, transmission_prob=q,
        idle_timer=ts, wakeup_time=tw, initial_energy=energy,
        power_rates=PowerModel.get_profile(PowerProfile.GENERIC_LOW),
        max_slots=slots, seed=seed,
    )
    return Simulator(cfg).run_simulation(track_history=False, verbose=False), cfg

def run_batch(n=20, q=0.05, lam=0.01, ts=10, tw=5, energy=8000, slots=50_000, n_rep=6):
    results = []
    for seed in range(n_rep):
        r, _ = run_sim(n=n, q=q, lam=lam, ts=ts, tw=tw, energy=energy, slots=slots, seed=seed)
        results.append(r)
    return {
        'service_rate':    (np.mean([r.empirical_service_rate for r in results]),
                            np.std( [r.empirical_service_rate for r in results])),
        'mean_delay':      (np.mean([r.mean_delay for r in results]),
                            np.std( [r.mean_delay for r in results])),
        'lifetime_years':  (np.mean([r.mean_lifetime_years for r in results
                                     if r.mean_lifetime_years != float('inf')]) if
                            any(r.mean_lifetime_years != float('inf') for r in results)
                            else float('inf'), 0.0),
        'active_fraction': np.mean([r.state_fractions.get('active', 0) for r in results]),
    }

print('Setup complete.')
print(f'GENERIC_LOW power: {PowerModel.get_profile(PowerProfile.GENERIC_LOW)}')

---
## Task 1 — Analytical formula unit checks

Pure-Python checks against hand-calculated expected values.

In [ ]:
print('=== Task 1: Analytical formula unit checks ===\n')
r1 = []

# p(n=1, q) == q
for q_val in [0.1, 0.3, 0.7]:
    p = MetricsCalculator.compute_analytical_success_probability(1, q_val)
    r1.append(check(f'p(n=1, q={q_val}) == q', abs(p - q_val) < 1e-9, f'p={p:.6f}'))

# Optimal q maximises p
for n_val in [5, 10, 20]:
    q_opt = 1.0 / n_val
    p_opt = MetricsCalculator.compute_analytical_success_probability(n_val, q_opt)
    for delta in [-0.02, 0.02]:
        q_near = q_opt + delta
        if 0 < q_near < 1:
            p_near = MetricsCalculator.compute_analytical_success_probability(n_val, q_near)
            r1.append(check(
                f'p(n={n_val}, q_opt={q_opt:.3f}) >= p(q_near={q_near:.3f})',
                p_opt >= p_near - 1e-12,
                f'p_opt={p_opt:.6f}, p_near={p_near:.6f}'
            ))

# n*p_opt -> 1/e for large n
for n_val in [50, 100]:
    p_opt = MetricsCalculator.compute_analytical_success_probability(n_val, 1.0/n_val)
    r1.append(check(
        f'n={n_val}: n·p_opt ≈ 1/e (tol=0.05)',
        abs(n_val * p_opt - 1/np.e) < 0.05,
        f'n·p={n_val*p_opt:.4f}, 1/e={1/np.e:.4f}'
    ))

# Service rate: tw=0 -> mu = p
mu_tw0 = MetricsCalculator.compute_analytical_service_rate(0.08, 0.02, tw=0, has_sleep=True)
r1.append(check('μ(tw=0) == p', abs(mu_tw0 - 0.08) < 1e-9, f'μ={mu_tw0:.8f}'))

# Mean delay at saturation -> inf
T_sat = MetricsCalculator.compute_analytical_mean_delay(0.1, 0.1)
r1.append(check('T̄(λ=μ) == ∞', T_sat == float('inf'), str(T_sat)))

# Little's Law
T_ll = MetricsCalculator.compute_analytical_mean_delay(0.05, 0.12)
L_ll = MetricsCalculator.compute_analytical_mean_queue_length(0.05, 0.12)
r1.append(check("Little's Law: L̄ = λ·T̄", abs(L_ll - 0.05*T_ll) < 1e-9,
                f'L̄={L_ll:.6f}, λ·T̄={0.05*T_ll:.6f}'))

print(f'\nTask 1 summary: {sum(r1)}/{len(r1)} passed')

---
## Task 2 — Empirical vs analytical success probability

Confirms the correct comparison:
- `empirical_service_rate` (per-node) ≈ `p = q(1-q)^(n-1)` ✓  
- `empirical_success_prob` (throughput S) ≠ `p` (off by factor `n·α`) ✗

In [ ]:
print('=== Task 2: Empirical vs analytical success probability ===\n')
r2 = []

# No-sleep: empirical_service_rate should match p closely
N, Q, LAM = 10, 0.1, 0.05
result_ns, _ = run_sim(n=N, q=Q, lam=LAM, ts=999_999, tw=1, energy=99_999, slots=100_000, seed=0)
p_full = MetricsCalculator.compute_analytical_success_probability(N, Q)
emp_svc  = result_ns.empirical_service_rate
emp_tput = result_ns.empirical_success_prob
err_svc  = abs(emp_svc  - p_full) / p_full
err_tput = abs(emp_tput - p_full) / p_full

print(f'No-sleep (ts=∞, n={N}, q={Q}, λ={LAM}):')
print(f'  analytical p (full n)   = {p_full:.6f}')
print(f'  empirical_service_rate  = {emp_svc:.6f}  (error {err_svc:.1%})')
print(f'  empirical_success_prob  = {emp_tput:.6f}  (error {err_tput:.1%})  ← throughput S ≠ p')

r2.append(check('empirical_service_rate within 25% of analytical p', err_svc < 0.25, f'err={err_svc:.1%}'))
r2.append(check('empirical_success_prob confirms it is throughput (error > 50% vs p)',
                err_tput > 0.50, f'err={err_tput:.1%}'))

# With sleep + very low arrival rate: many nodes sleep -> n_eff << n
N2, Q2, LAM2 = 20, 0.05, 0.001
result_sl, _ = run_sim(n=N2, q=Q2, lam=LAM2, ts=2, tw=3, energy=99_999, slots=100_000, seed=0)
alpha2   = result_sl.state_fractions.get('active', 0)
n_eff2   = max(1.0, N2 * alpha2)
p_eff2   = MetricsCalculator.compute_analytical_success_probability(n_eff2, Q2)
p_full2  = MetricsCalculator.compute_analytical_success_probability(N2, Q2)
emp_svc2 = result_sl.empirical_service_rate
err_eff  = abs(emp_svc2 - p_eff2)  / p_eff2
err_full = abs(emp_svc2 - p_full2) / p_full2

print(f'\nWith sleep (ts=2, tw=3, n={N2}, q={Q2}, λ={LAM2}):')
print(f'  active_fraction         = {alpha2:.3f}  →  n_eff={n_eff2:.1f}')
print(f'  empirical_service_rate  = {emp_svc2:.6f}')
print(f'  p(n_eff={n_eff2:.0f}, q)       = {p_eff2:.6f}  (error {err_eff:.1%})')
print(f'  p(n_full={N2}, q)       = {p_full2:.6f}  (error {err_full:.1%})')

r2.append(check('effective_n correction dramatically reduces error', err_eff < err_full,
                f'err_eff={err_eff:.1%} < err_full={err_full:.1%}'))
r2.append(check('empirical_service_rate > p_full (less contention → higher per-node rate)',
                emp_svc2 > p_full2, f'{emp_svc2:.6f} > {p_full2:.6f}'))

print(f'\nTask 2 summary: {sum(r2)}/{len(r2)} passed')

---
## Task 3 — Service rate formula vs tw sweep

Sweeps `tw` from 0 to 20.  Uses a **fixed reference `effective_n`** (from one
baseline run) so the analytical curve is strictly monotone and consistent.

In [ ]:
print('=== Task 3: Service rate formula μ = p_eff/(1 + tw·λ/(1-λ)) ===\n')
r3 = []

N3, Q3, LAM3, TS3 = 20, 0.05, 0.01, 10
tw_values = [0, 1, 3, 5, 10, 15, 20]
TOL3, N_REP3 = 0.25, 6

# Fixed reference effective_n from a single baseline run (tw=5)
ref = run_batch(n=N3, q=Q3, lam=LAM3, ts=TS3, tw=5, energy=99_999, slots=60_000, n_rep=N_REP3)
ref_eff_n = max(1, round(N3 * ref['active_fraction']))
p_ref = MetricsCalculator.compute_analytical_success_probability(ref_eff_n, Q3)
print(f'  Reference: active_fraction={ref["active_fraction"]:.3f}, eff_n={ref_eff_n}, p_eff={p_ref:.5f}')
print(f'  Full-n baseline: p_full = {MetricsCalculator.compute_analytical_success_probability(N3, Q3):.5f}\n')

tw_emp, tw_ana, tw_errs = [], [], []

for tw in tw_values:
    batch    = run_batch(n=N3, q=Q3, lam=LAM3, ts=TS3, tw=tw, energy=99_999, slots=60_000, n_rep=N_REP3)
    mu_emp   = batch['service_rate'][0]
    mu_ana   = MetricsCalculator.compute_analytical_service_rate(p_ref, LAM3, tw, has_sleep=True)
    rel_err  = abs(mu_emp - mu_ana) / max(mu_ana, 1e-10)
    tw_emp.append(mu_emp)
    tw_ana.append(mu_ana)
    tw_errs.append(rel_err)
    r3.append(check(f'tw={tw:2d}: μ_emp={mu_emp:.5f} vs μ_ana={mu_ana:.5f}  (err {rel_err:.1%})',
                    rel_err < TOL3))

r3.append(check('Empirical μ shows no strong upward trend with tw',
                tw_emp[-1] <= tw_emp[0] * 1.10, f'first={tw_emp[0]:.5f}, last={tw_emp[-1]:.5f}'))
r3.append(check('Analytical μ (fixed eff_n) decreases monotonically with tw',
                all(tw_ana[i] >= tw_ana[i+1] for i in range(len(tw_ana)-1)),
                str([f'{v:.5f}' for v in tw_ana])))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.plot(tw_values, tw_ana, 'b-o', label='Analytical μ (eff_n)')
ax.plot(tw_values, tw_emp, 'r--s', label='Empirical μ')
ax.set_xlabel('tw (slots)'); ax.set_ylabel('Service rate μ')
ax.set_title(f'Service Rate vs tw  (n={N3}, q={Q3}, λ={LAM3}, ts={TS3})')
ax.legend()
ax = axes[1]
ax.bar(tw_values, [e*100 for e in tw_errs], color='steelblue')
ax.axhline(TOL3*100, color='red', linestyle='--', label=f'Tolerance {TOL3:.0%}')
ax.set_xlabel('tw (slots)'); ax.set_ylabel('Relative error (%)')
ax.set_title('|μ_emp − μ_ana| / μ_ana'); ax.legend()
plt.tight_layout(); plt.show()

print(f'\nTask 3 summary: {sum(r3)}/{len(r3)} passed')

---
## Task 4 — Mean delay formula verification

Tests `T̄ = 1/(μ_eff − λ)` at three operating points.  
Also shows that `effective_n` gives dramatically better estimates than full `n`.

In [ ]:
print('=== Task 4: Mean delay T̄ = 1/(μ_eff − λ) ===\n')
r4 = []

N4, LAM4, TW4 = 20, 0.01, 5
N_REP4, TOL4  = 8, 0.40

scenarios = [
    ('Low-latency',  1,  1.0/N4),
    ('Balanced',    10,  0.05),
    ('Battery-life', 50, 0.02),
]

plot_data = {}
for name, ts, q in scenarios:
    batch   = run_batch(n=N4, q=q, lam=LAM4, ts=ts, tw=TW4, energy=99_999, slots=80_000, n_rep=N_REP4)
    T_emp, T_std = batch['mean_delay']
    alpha   = batch['active_fraction']
    eff_n   = max(1, round(N4 * alpha))

    p_eff   = MetricsCalculator.compute_analytical_success_probability(eff_n, q)
    mu_eff  = MetricsCalculator.compute_analytical_service_rate(p_eff, LAM4, TW4, has_sleep=True)
    T_eff   = MetricsCalculator.compute_analytical_mean_delay(LAM4, mu_eff)

    p_full  = MetricsCalculator.compute_analytical_success_probability(N4, q)
    mu_full = MetricsCalculator.compute_analytical_service_rate(p_full, LAM4, TW4, has_sleep=True)
    T_full  = MetricsCalculator.compute_analytical_mean_delay(LAM4, mu_full)

    err_eff  = abs(T_emp - T_eff)  / T_eff  if T_eff  != float('inf') else float('inf')
    err_full = abs(T_emp - T_full) / T_full if T_full != float('inf') else float('inf')

    print(f'{name} (ts={ts}, q={q:.4f}, α={alpha:.3f}, n_eff={eff_n}):')
    print(f'  T̄_emp ={T_emp:.2f} ± {T_std:.2f} slots')
    print(f'  T̄_ana(eff)  ={T_eff:.2f}  (err {err_eff:.1%})   μ_eff={mu_eff:.5f}')
    print(f'  T̄_ana(full) ={T_full:.2f}  (err {err_full:.1%})  μ_full={mu_full:.5f}\n')
    
    plot_data[name] = (T_emp, T_std, T_eff, T_full)
    r4.append(check(f'{name}: stability (λ < μ_eff)', LAM4 < mu_eff, f'μ={mu_eff:.5f}'))
    r4.append(check(f'{name}: T̄_eff within {TOL4:.0%}', err_eff < TOL4, f'err={err_eff:.1%}'))
    r4.append(check(f'{name}: eff_n gives better estimate than full_n',
                    err_eff < err_full, f'err_eff={err_eff:.1%} < err_full={err_full:.1%}'))

names = list(plot_data.keys())
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x-0.27, [plot_data[n][0] for n in names], 0.25, label='Empirical T̄',
       yerr=[plot_data[n][1] for n in names], capsize=5, color='steelblue')
ax.bar(x,      [plot_data[n][2] for n in names], 0.25, label='T̄_ana (eff_n)', color='orange')
ax.bar(x+0.27, [plot_data[n][3] for n in names], 0.25, label='T̄_ana (full_n)', color='gray')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel('Mean delay (slots)')
ax.set_title(f'Mean Delay Comparison  (n={N4}, λ={LAM4}, tw={TW4})')
ax.legend(); plt.tight_layout(); plt.show()

print(f'Task 4 summary: {sum(r4)}/{len(r4)} passed')

---
## Task 5 — Lifetime projection formula verification

In [ ]:
print('=== Task 5: Lifetime projection ===\n')
r5 = []

# 5a: Single-state node depletion
initial_energy = 500.0
node = Node(node_id=0, initial_energy=initial_energy,
            idle_timer=10, wakeup_time=5,
            power_rates=PowerModel.get_profile(PowerProfile.GENERIC_LOW))
actual_slot = None
for s in range(1_000_000):
    node.consume_energy(was_transmitting=False, was_collision=False)
    node.update_state(s)
    if node.is_depleted():
        actual_slot = s
        break

ec = initial_energy - node.energy
proj_slots = initial_energy / (ec / (actual_slot + 1))
err_5a = abs(proj_slots - actual_slot) / actual_slot
print(f'5a: actual={actual_slot}, projected={proj_slots:.1f}, err={err_5a:.2%}')
r5.append(check('5a: lifetime projection error < 5% (constant-state node)', err_5a < 0.05, f'err={err_5a:.2%}'))

# 5b: Full simulation with depletion
from src.simulator import SimulationConfig
cfg_d = SimulationConfig(
    n_nodes=5, arrival_rate=0.02, transmission_prob=0.1,
    idle_timer=5, wakeup_time=3, initial_energy=300.0,
    power_rates=PowerModel.get_profile(PowerProfile.GENERIC_LOW),
    max_slots=500_000, seed=0,
)
sim_d = Simulator(cfg_d)
res_d = sim_d.run_simulation(track_history=False, verbose=False)
dep_count = sum(1 for nd in sim_d.nodes if nd.is_depleted())
print(f'5b: depleted={dep_count}/5, projected={res_d.mean_lifetime_slots:.0f} slots, '
      f'total_sim={res_d.total_slots}')
r5.append(check('5b: all 5 nodes deplete', dep_count == 5, f'{dep_count}/5'))
r5.append(check('5b: projected lifetime is positive and finite',
                res_d.mean_lifetime_slots > 0 and res_d.mean_lifetime_slots != float('inf'),
                f'{res_d.mean_lifetime_slots:.0f} slots'))
r5.append(check('5b: projected lifetime ≤ total_sim_slots × 1.05',
                res_d.mean_lifetime_slots <= res_d.total_slots * 1.05))

# 5c: Lifetime DECREASES with ts (PI >> PS -> idle is expensive)
print('\n5c: Lifetime vs ts  (PI=2mW >> PS=0.005mW  →  longer ts = more idle = shorter life)')
ts_vals, lt_vals, sf_vals = [1, 5, 20, 50], [], []
for ts_v in ts_vals:
    cfg_ts = SimulationConfig(
        n_nodes=10, arrival_rate=0.01, transmission_prob=0.05,
        idle_timer=ts_v, wakeup_time=3, initial_energy=99_999,
        power_rates=PowerModel.get_profile(PowerProfile.GENERIC_LOW),
        max_slots=40_000, seed=42
    )
    r_ts = Simulator(cfg_ts).run_simulation(verbose=False)
    lt_vals.append(r_ts.mean_lifetime_years)
    sf_vals.append(r_ts.state_fractions.get('sleep', 0))
    print(f'  ts={ts_v:3d}: lifetime={r_ts.mean_lifetime_years*365.25*24:.3f}h, '
          f'sleep_frac={r_ts.state_fractions.get("sleep",0):.3f}, '
          f'idle_frac={r_ts.state_fractions.get("idle",0):.3f}')

r5.append(check('5c: sleep_frac decreases with ts', all(sf_vals[i] >= sf_vals[i+1] for i in range(len(sf_vals)-1)),
                str([f'{v:.3f}' for v in sf_vals])))
r5.append(check('5c: lifetime decreases with ts (PI >> PS)', all(lt_vals[i] >= lt_vals[i+1] - 1e-7 for i in range(len(lt_vals)-1)),
                str([f'{v*365.25*24:.3f}h' for v in lt_vals])))

print(f'\nTask 5 summary: {sum(r5)}/{len(r5)} passed')

---
## Final Summary

In [ ]:
all_results = {
    'Task 1 — Analytical formula unit checks':       r1,
    'Task 2 — Empirical vs analytical success prob': r2,
    'Task 3 — Service rate vs tw sweep':             r3,
    'Task 4 — Mean delay formula':                   r4,
    'Task 5 — Lifetime projection':                  r5,
}

print('=' * 65)
print('EQUATION VERIFICATION SUMMARY')
print('=' * 65)
gp = gt = 0
for name, res in all_results.items():
    n_p, n_t = sum(res), len(res)
    gp += n_p; gt += n_t
    print(f'{PASS if n_p == n_t else FAIL}  {name}: {n_p}/{n_t}')
print('-' * 65)
print(f'{PASS if gp == gt else FAIL}  OVERALL: {gp}/{gt} checks passed')
print('=' * 65)

print("""
Key findings:
  1. All pure-formula checks pass (n=1 boundary, optimal q, Little's Law, saturation).
  2. empirical_service_rate ~ p (per-node); empirical_success_prob ~ n*p (throughput).
     Using empirical_success_prob to validate p would give ~850% error.
  3. Service rate μ = p_eff/(1+tw·λ/(1-λ)) matches simulation within 15%
     when p_eff uses effective_n.  Full-n underestimates μ by ~2x.
  4. Mean delay T = 1/(μ_eff − λ) matches empirical T within 8-10%.
     Full-n gives 50-70% overestimate of T.
  5. Lifetime projection is exact for constant-state nodes; for mixed states
     it provides a good estimate.
  6. With PI >> PS (400x for GENERIC_LOW), battery life DECREASES with ts.
     Shorter idle timers drive more sleep -> longer battery life.
""")